# 01 - EDA Current Stress
EDA + tracking MLflow + output lokal.

In [17]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import importlib.util
from io import StringIO

import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "nostressia-machine-learning" / "Current-Stress" / "notebooks" / "experiments",
]
UTILS_PATH = next((d / "mlflow_utils.py" for d in CANDIDATE_DIRS if (d / "mlflow_utils.py").exists()), None)
if UTILS_PATH is None:
    raise FileNotFoundError("mlflow_utils.py tidak ditemukan. Pastikan notebook dijalankan dari root repo atau folder experiments.")

spec = importlib.util.spec_from_file_location("cs_mlflow_utils", UTILS_PATH)
cs_utils = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(cs_utils)

RANDOM_STATE = cs_utils.RANDOM_STATE
configure_mlflow = cs_utils.configure_mlflow
set_seeds = cs_utils.set_seeds
load_current_stress_dataset = cs_utils.load_current_stress_dataset
get_dataset_path = cs_utils.get_dataset_path
log_run_metadata = cs_utils.log_run_metadata
create_local_output_dir = cs_utils.create_local_output_dir

repo_root = configure_mlflow()
set_seeds(RANDOM_STATE)


MlflowException: Cannot set a deleted experiment 'Current Stress' as the active experiment. You can restore the experiment, or permanently delete the experiment to create a new one.

In [ ]:
NOTEBOOK_NAME = "01_eda_current_stress.ipynb"
raw_df, feature_df, y = load_current_stress_dataset(repo_root)
dataset_source = str(get_dataset_path(repo_root))

with mlflow.start_run(run_name="EDA - Current Stress") as run:
    log_run_metadata(
        run_description="EDA current stress; dataset=current_stress_v1; split metadata=80/20",
        tags={"features": "all", "model": "EDA"},
        params={"stage": "eda", "rows": len(raw_df), "columns": raw_df.shape[1]},
        dataset_df=raw_df.assign(target=y.values),
        dataset_context="eda",
        dataset_source=dataset_source,
    )

    local_output_dir = create_local_output_dir(repo_root, NOTEBOOK_NAME, run.info.run_id)
    print(f"[ipynb-result] Output folder: {local_output_dir}")

    print("Shape of the dataset:", raw_df.shape)
    display(raw_df.head())

    print("\nDataset Information:")
    info_buffer = StringIO()
    raw_df.info(buf=info_buffer)
    print(info_buffer.getvalue())

    print("\nStatistical Summary:")
    describe_df = raw_df.describe(include="all").transpose()
    display(describe_df)

    missing_total = int(raw_df.isna().sum().sum())
    duplicated_total = int(raw_df.duplicated().sum())
    print(f"\nMissing values: {missing_total}")
    print(f"Duplicated values: {duplicated_total}")

    print("\nUnique Values in Each Column:")
    unique_counts = raw_df.nunique()
    print(unique_counts)

    numerical_columns = raw_df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    non_numerical_columns = raw_df.select_dtypes(exclude=["int64", "float64"]).columns.tolist()
    print("\nNumerical Columns:", numerical_columns)
    print("Categorical Columns:", non_numerical_columns)

    describe_df.to_csv(local_output_dir / "eda_describe.csv")
    raw_df.isna().sum().rename("missing_count").to_csv(local_output_dir / "missing_values.csv")
    unique_counts.rename("unique_count").to_csv(local_output_dir / "unique_values.csv")

    # Distribution plot: count + pie
    column_name = "Stress_Level"
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    stress_counts = raw_df[column_name].value_counts()
    sns.countplot(y=column_name, data=raw_df, palette="muted", order=stress_counts.index, ax=axes[0])
    axes[0].set_title(f"Distribution of {column_name}")
    for p in axes[0].patches:
        axes[0].annotate(f"{int(p.get_width())}", (p.get_width(), p.get_y() + p.get_height()/2), ha="left", va="center", xytext=(5,0), textcoords="offset points")

    stress_counts.plot.pie(
        autopct="%1.1f%%",
        colors=sns.color_palette("muted"),
        startangle=90,
        explode=[0.05]*stress_counts.shape[0],
        ax=axes[1],
    )
    axes[1].set_title(f"Percentage Distribution of {column_name}")
    axes[1].set_ylabel("")
    fig.tight_layout()
    fig.savefig(local_output_dir / "stress_label_distribution.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Numeric distribution histograms
    hist_cols = [
        "Study_Hours_Per_Day",
        "Extracurricular_Hours_Per_Day",
        "Sleep_Hours_Per_Day",
        "Social_Hours_Per_Day",
        "Physical_Activity_Hours_Per_Day",
        "GPA",
    ]
    fig, axes = plt.subplots(3, 2, figsize=(10, 12))
    for ax, col in zip(axes.flatten(), hist_cols):
        sns.histplot(raw_df[col], kde=True, bins=10, ax=ax, color=sns.color_palette("muted")[0])
        ax.set_title(f"{col} Distribution")
    fig.tight_layout()
    fig.savefig(local_output_dir / "numeric_histograms.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Boxplots + summary text
    fig, axes = plt.subplots(3, 2, figsize=(10, 12))
    for ax, col in zip(axes.flatten(), hist_cols):
        sns.boxplot(x=raw_df[col], ax=ax, color=sns.color_palette("muted")[1])
        ax.set_title(f"{col} Boxplot")
        print(f"\nSummary Statistics for {col}:")
        print(raw_df[col].describe())
    fig.tight_layout()
    fig.savefig(local_output_dir / "numeric_boxplots.png", dpi=150)
    plt.show()
    plt.close(fig)

    # Correlation heatmap
    corr = raw_df[numerical_columns].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title("Correlation Heatmap of Numerical Features")
    plt.tight_layout()
    plt.savefig(local_output_dir / "correlation_heatmap.png", dpi=150)
    plt.show()
    plt.close()

    low_gpa_students = raw_df[raw_df["GPA"] <= 2.5][["GPA", "Study_Hours_Per_Day", "Physical_Activity_Hours_Per_Day", "Stress_Level"]]
    high_gpa_students = raw_df[raw_df["GPA"] >= 3.5][["GPA", "Study_Hours_Per_Day", "Physical_Activity_Hours_Per_Day", "Stress_Level"]]
    print("\n----- Students with Low GPA (<= 2.5) -----")
    display(low_gpa_students.head(20))
    print("\n----- Students with High GPA (>= 3.5) -----")
    display(high_gpa_students.head(20))

    mlflow.log_artifacts(str(local_output_dir), artifact_path="local_outputs")

    print("[ipynb-result] Saved files:")
    for saved_file in sorted(local_output_dir.glob("*")):
        print("-", saved_file.name)

    # Inline previews from saved artifacts
    display(Image(filename=str(local_output_dir / "stress_label_distribution.png")))
    display(Image(filename=str(local_output_dir / "numeric_histograms.png")))
    display(Image(filename=str(local_output_dir / "numeric_boxplots.png")))
    display(Image(filename=str(local_output_dir / "correlation_heatmap.png")))

    print("Run ID:", run.info.run_id)
    print("Local output:", local_output_dir)

